In [7]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

aerospace_data = [
    ("A1", "Falcon 9", "Rocket", "active", "C1"),
    ("A2", "Starship", "Rocket", "active", "C1"),
    ("A3", "Hubble", "Telescope", "active", "C2"),
    ("A4", "Galileo", "Satellite", "inactive", "C3"),
    ("A5", "Voyager 1", "Probe", "active", "C3"),
]

aerospace_columns = ["id", "name", "type", "status", "company_id"]
aerospace_df = spark.createDataFrame(aerospace_data, aerospace_columns)

company_data = [
    ("C1", "SpaceX", "USA"),
    ("C2", "NASA", "USA"),
    ("C3", "European Space Agency", "Europe"),
]

company_columns = ["id", "name", "country"]
company_df = spark.createDataFrame(company_data, company_columns)

aerospace_df.show()
company_df.show()

+---+---------+---------+--------+----------+
| id|     name|     type|  status|company_id|
+---+---------+---------+--------+----------+
| A1| Falcon 9|   Rocket|  active|        C1|
| A2| Starship|   Rocket|  active|        C1|
| A3|   Hubble|Telescope|  active|        C2|
| A4|  Galileo|Satellite|inactive|        C3|
| A5|Voyager 1|    Probe|  active|        C3|
+---+---------+---------+--------+----------+

+---+--------------------+-------+
| id|                name|country|
+---+--------------------+-------+
| C1|              SpaceX|    USA|
| C2|                NASA|    USA|
| C3|European Space Ag...| Europe|
+---+--------------------+-------+



In [9]:
company_df.join(aerospace_df, aerospace_df.company_id == company_df.id).withColumn(
    "status_label",
    when(
        (col("status") == "active") & (col("country") == "USA"), lit("Domestic Active")
    )
    .when(
        (col("status") == "active") & (col("country") != "USA"), lit("Foreign Active")
    )
    .otherwise(lit("inactive")),
).show()

+---+--------------------+-------+---+---------+---------+--------+----------+---------------+
| id|                name|country| id|     name|     type|  status|company_id|   status_label|
+---+--------------------+-------+---+---------+---------+--------+----------+---------------+
| C1|              SpaceX|    USA| A1| Falcon 9|   Rocket|  active|        C1|Domestic Active|
| C1|              SpaceX|    USA| A2| Starship|   Rocket|  active|        C1|Domestic Active|
| C2|                NASA|    USA| A3|   Hubble|Telescope|  active|        C2|Domestic Active|
| C3|European Space Ag...| Europe| A4|  Galileo|Satellite|inactive|        C3|       inactive|
| C3|European Space Ag...| Europe| A5|Voyager 1|    Probe|  active|        C3| Foreign Active|
+---+--------------------+-------+---+---------+---------+--------+----------+---------------+

